# 教师参考答案｜请勿随学生版分发

逐题对应学生练习。依赖与路径同学生版；结果用于教师核对，不代表学生完成或课堂已实施。

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# 自动定位公共数据目录：可从本周根目录或代码所在子目录运行。
# 脚本以自身位置为起点，Notebook以当前工作目录为起点。
BASE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
if not (BASE / 'data' / 'salesdata.csv').is_file():
    BASE = BASE.parent
DATA = BASE / 'data'
OUTPUT = BASE / 'outputs'
if not (DATA / 'salesdata.csv').is_file():
    raise FileNotFoundError('请在第五周根目录或上课文件、练习文件、选做内容目录中运行。')
OUTPUT.mkdir(exist_ok=True)
print('pandas 版本：', pd.__version__)

pandas 版本： 3.1.0


## 第1题｜读取与结构（20分）

读取salesdata.csv为df；显示前3行、shape、列名。分别取“销售量/个”单列Series和单列DataFrame，打印形状。用自己的话解释1351与5分别代表什么。提示：read_csv、head、两层方括号。

In [2]:
df = pd.read_csv(DATA / 'salesdata.csv', encoding='utf-8')
display(df.head(3))
print(df.shape, df.columns.tolist())
print(df['销售量/个'].shape, df[['销售量/个']].shape)

,商品ID,商品类别,订单时间,销售量/个,销售价格/元
0,10004328,类别A,2023/3/11 4:51,3,147
1,10000346,类别A,2023/3/9 4:16,8,239
2,10001192,类别A,2023/3/8 22:21,4,53


(1351, 5) ['商品ID', '商品类别', '订单时间', '销售量/个', '销售价格/元']
(1351,) (1351, 1)


参考解释：表中1351条记录、5个字段。Series形状(1351,)，单列DataFrame形状(1351, 1)。

## 第2题｜索引与条件（30分）

使用loc选标签2—6的销售量、销售价格；用iloc取得相同结果。再筛选整数商品ID为10000019的两列。解释iloc为什么右端写7，说明商品ID筛选后有几条记录。提示：不要把布尔Series交给iloc。

In [3]:
by_label = df.loc[2:6, ['销售量/个', '销售价格/元']]
by_position = df.iloc[2:7, [3, 4]]
display(by_label)
assert by_label.equals(by_position)
selected = df.loc[df['商品ID'] == 10000019, ['销售量/个', '销售价格/元']]
display(selected)
print('筛选条数：', len(selected))

,销售量/个,销售价格/元
2,4,53
3,3,43
4,1,13
5,2,16
6,2,14


,销售量/个,销售价格/元
815,2,25
997,3,38


筛选条数： 2


参考解释：loc标签切片包含右端6；iloc排除右端，所以写7。默认未删行索引下两者选中同5行；该商品筛得2条记录。

## 第3题｜副本修改与基线统计（25分）

复制df为practice，把类别B改为C；添加“练习标记”列，值为1，再删除该列（不使用inplace）。检查df的类别没有改变。对原df求销售价格均值、中位数和类别频数，写明平均的统计对象与单位。

In [4]:
practice = df.copy()
practice.loc[practice['商品类别'] == '类别B', '商品类别'] = '类别C'
practice['练习标记'] = 1
practice = practice.drop(columns='练习标记')
print('原表类别：', df['商品类别'].value_counts().to_dict())
print('修改副本类别：', practice['商品类别'].value_counts().to_dict())
print(df['销售价格/元'].mean(), df['销售价格/元'].median())

原表类别： {'类别B': 724, '类别A': 627}
修改副本类别： {'类别C': 724, '类别A': 627}
82.23316062176166 51.0


参考解释：原表A627、B724，副本A627、C724。价格均值约82.23元，中位数51元，是1351条销售记录的价格简单平均，不是销量加权价格。

## 第4题｜产量迁移与解释（25分）

读取工业产品产量数据.csv；用下方提供的条件选出2023年原煤记录。自行统计记录数、非缺失数、缺失数和均值。写2句话说明为什么不能把缺失填0、为什么不能对所有产品产量直接求总量。

筛选条件已提供，复杂日期与分组计算不作为本题要求。

In [5]:
industry = pd.read_csv(DATA / '工业产品产量数据.csv', encoding='utf-8')
coal = industry.loc[(industry['工业产品名称'] == '原煤产量当期值/万吨') & industry['时间'].str.startswith('2023-')]
print(len(coal), coal['产量'].count(), coal['产量'].isna().sum())
print(coal['产量'].mean())

12 10 2
39439.79


参考解释：12条记录中10条有值、2条缺失；10个有效月份平均39439.79万吨。缺失原因未知，不能断言零产量；不同产品含不同计量单位且对象不可直接相加，不能给出混合单位总量。

## 拓展参考

In [6]:
selected.to_csv(OUTPUT / '练习筛选.csv', index=False, encoding='utf-8-sig')
readback = pd.read_csv(OUTPUT / '练习筛选.csv', encoding='utf-8-sig')
assert selected.reset_index(drop=True).equals(readback)
string_id = pd.read_csv(DATA / 'salesdata.csv', dtype={'商品ID': 'string'}, encoding='utf-8')
display(string_id.loc[string_id['商品ID'] == '10000019'])

,商品ID,商品类别,订单时间,销售量/个,销售价格/元
815,10000019,类别B,2022/9/9 9:49,2,25
997,10000019,类别B,2020/9/5 11:41,3,38


标识符用string可保留前导零；本例原数据已按整数存储，不人为补零。按string读取后比较值也要用字符串。